# Script for comparing the perfomance and scaling of Quadratic Annealing Optimizer with CPU and GPU backend

In [6]:
import torch
import torch.nn as nn
import torch
import copy

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import Logistic, QuadraticMLP
from src.utils import data_load_and_prep, build_sampler
from src.training import train
from src.newton_optimizer import NewtonOptimizer
from tqdm import tqdm

EXPERIMENT_NAME = "sim-bachend-comparison"

## Scaling with number of num_reads/replicas on Iris dataset

In [5]:
train_loader, test_loader = data_load_and_prep("iris", test_size=0.3, random_state=42, batch_size='full')

In [7]:
def run_experiments(loss_fn: nn.Module,
                    classical_device: str,
                    num_reads: list[int],
                    mode: str,
                    epochs: int=50,
) -> None:
    
    for num_read in tqdm(num_reads):
        torch.manual_seed(42)
        model = QuadraticMLP(input_dim=4, hidden_dim=[32, 16], output_dim=3)
        sampler = build_sampler(mode=mode)
        optimizer = QuadraticAnnealingOptimizer(
            sampler=sampler,
            model=model,
            device=classical_device,
            num_reads=num_read,
            step_size=0.05,
            subset_size=12,
        )
        train(model, 
              train_loader=train_loader,
              test_loader=test_loader,
              optimizer=optimizer,
              loss_fn=loss_fn,
              c_device=classical_device,
              epochs=epochs,
              experiment_name=EXPERIMENT_NAME,
              run_name=f"{mode}_num_reads_{num_read}",
              )

In [12]:
num_reads = [1000_000, 10_000_000]
loss_fn = nn.CrossEntropyLoss()

In [10]:
# CPU
run_experiments(loss_fn=loss_fn,
                classical_device="cpu",
                num_reads=num_reads,
                mode="simulated",
)

  0%|          | 0/8 [00:00<?, ?it/s]/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/08 14:12:54 INFO mlflow.tracking.fluent: Experiment with name 'sim-bachend-comparison' does not exist. Creating a new experiment.


Epoch 000 | train_loss=1.0434 | train_acc=0.333 | test_loss=1.0618 | test_acc=0.333 | 
Epoch 005 | train_loss=0.7475 | train_acc=0.676 | test_loss=0.7798 | test_acc=0.689 | 
Epoch 010 | train_loss=0.5377 | train_acc=0.848 | test_loss=0.5817 | test_acc=0.756 | 
Epoch 015 | train_loss=0.3972 | train_acc=0.886 | test_loss=0.4730 | test_acc=0.778 | 
Epoch 020 | train_loss=0.2784 | train_acc=0.924 | test_loss=0.3965 | test_acc=0.844 | 
Epoch 025 | train_loss=0.1789 | train_acc=0.981 | test_loss=0.2848 | test_acc=0.867 | 
Epoch 030 | train_loss=0.1081 | train_acc=0.971 | test_loss=0.1705 | test_acc=0.933 | 
Epoch 035 | train_loss=0.0669 | train_acc=0.981 | test_loss=0.1362 | test_acc=0.911 | 
Epoch 040 | train_loss=0.0431 | train_acc=0.990 | test_loss=0.1735 | test_acc=0.911 | 


2026/04/08 14:12:56 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 045 | train_loss=0.0316 | train_acc=1.000 | test_loss=0.1809 | test_acc=0.911 | 
Epoch 049 | train_loss=0.0252 | train_acc=1.000 | test_loss=0.2063 | test_acc=0.889 | 


 12%|█▎        | 1/8 [00:04<00:32,  4.67s/it]

...

 88%|████████▊ | 7/8 [04:37<01:19, 79.05s/it]

Epoch 000 | train_loss=1.0434 | train_acc=0.333 | test_loss=1.0618 | test_acc=0.333 | 
Epoch 005 | train_loss=0.7475 | train_acc=0.676 | test_loss=0.7798 | test_acc=0.689 | 
Epoch 010 | train_loss=0.5377 | train_acc=0.848 | test_loss=0.5817 | test_acc=0.756 | 
Epoch 015 | train_loss=0.3972 | train_acc=0.886 | test_loss=0.4730 | test_acc=0.778 | 
Epoch 020 | train_loss=0.2784 | train_acc=0.924 | test_loss=0.3965 | test_acc=0.844 | 
Epoch 025 | train_loss=0.1789 | train_acc=0.981 | test_loss=0.2848 | test_acc=0.867 | 
Epoch 030 | train_loss=0.1081 | train_acc=0.971 | test_loss=0.1705 | test_acc=0.933 | 
Epoch 035 | train_loss=0.0669 | train_acc=0.981 | test_loss=0.1362 | test_acc=0.911 | 
Epoch 040 | train_loss=0.0431 | train_acc=0.990 | test_loss=0.1735 | test_acc=0.911 | 
Epoch 045 | train_loss=0.0316 | train_acc=1.000 | test_loss=0.1809 | test_acc=0.911 | 


2026/04/08 14:24:04 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=0.0252 | train_acc=1.000 | test_loss=0.2063 | test_acc=0.889 | 


100%|██████████| 8/8 [11:11<00:00, 83.95s/it] 


In [13]:
# GPU
run_experiments(loss_fn=loss_fn,
                classical_device="cuda",
                num_reads=num_reads,
                mode="gpu_simulated",
)

  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 000 | train_loss=1.0434 | train_acc=0.333 | test_loss=1.0618 | test_acc=0.333 | 
Epoch 005 | train_loss=0.7475 | train_acc=0.676 | test_loss=0.7798 | test_acc=0.689 | 
Epoch 010 | train_loss=0.5377 | train_acc=0.848 | test_loss=0.5817 | test_acc=0.756 | 
Epoch 015 | train_loss=0.3972 | train_acc=0.886 | test_loss=0.4730 | test_acc=0.778 | 
Epoch 020 | train_loss=0.2784 | train_acc=0.924 | test_loss=0.3965 | test_acc=0.844 | 
Epoch 025 | train_loss=0.1789 | train_acc=0.981 | test_loss=0.2848 | test_acc=0.867 | 
Epoch 030 | train_loss=0.1081 | train_acc=0.971 | test_loss=0.1705 | test_acc=0.933 | 
Epoch 035 | train_loss=0.0669 | train_acc=0.981 | test_loss=0.1362 | test_acc=0.911 | 
Epoch 040 | train_loss=0.0431 | train_acc=0.990 | test_loss=0.1735 | test_acc=0.911 | 
Epoch 045 | train_loss=0.0316 | train_acc=1.000 | test_loss=0.1809 | test_acc=0.911 | 


2026/04/08 15:02:02 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=0.0252 | train_acc=1.000 | test_loss=0.2063 | test_acc=0.889 | 


 50%|█████     | 1/2 [00:20<00:20, 20.48s/it]

Epoch 000 | train_loss=1.0434 | train_acc=0.333 | test_loss=1.0618 | test_acc=0.333 | 
Epoch 005 | train_loss=0.7475 | train_acc=0.676 | test_loss=0.7798 | test_acc=0.689 | 
Epoch 010 | train_loss=0.5377 | train_acc=0.848 | test_loss=0.5817 | test_acc=0.756 | 
Epoch 015 | train_loss=0.3972 | train_acc=0.886 | test_loss=0.4730 | test_acc=0.778 | 
Epoch 020 | train_loss=0.2784 | train_acc=0.924 | test_loss=0.3965 | test_acc=0.844 | 
Epoch 025 | train_loss=0.1789 | train_acc=0.981 | test_loss=0.2848 | test_acc=0.867 | 
Epoch 030 | train_loss=0.1081 | train_acc=0.971 | test_loss=0.1705 | test_acc=0.933 | 
Epoch 035 | train_loss=0.0669 | train_acc=0.981 | test_loss=0.1362 | test_acc=0.911 | 
Epoch 040 | train_loss=0.0431 | train_acc=0.990 | test_loss=0.1735 | test_acc=0.911 | 
Epoch 045 | train_loss=0.0316 | train_acc=1.000 | test_loss=0.1809 | test_acc=0.911 | 


2026/04/08 15:03:38 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=0.0252 | train_acc=1.000 | test_loss=0.2063 | test_acc=0.889 | 


100%|██████████| 2/2 [01:55<00:00, 57.78s/it]
